In [26]:
#Imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import datetime


In [27]:
#Read the file and pick the specific sheet that I want 
excelSheet = pd.ExcelFile('/Users/preciousajilore/Documents/GitHub/torchmtlr/notebooks/first_cleaned.xlsx')
print(excelSheet.sheet_names)

df = pd.read_excel(
                  excelSheet,         #  name of the excel file
                  sheet_name='Sheet1', #   name of the sheet you want to read
                  engine='openpyxl')

['Sheet1']


In [28]:
df["ER visits"]

0       yes
1        No
2        no
3        no
4       yes
       ... 
1982      0
1983    NaN
1984    NaN
1985      0
1986    NaN
Name: ER visits, Length: 1987, dtype: object

In [29]:
print(df["ER visits"].unique())

['yes' 'No' 'no'
 'yes(hematoma,celulitis of the penis,urethrocutaneous fistula)'
 'no (yes for CP/GERD at just under 90d)' 'Yes'
 'Yes - post op wound pain, uncomplicated, no infection' 'yes - UTI'
 'Yes, pre-op' 'No (only pre-op x 1)' 'yes - wound infection' nan
 'Yes, for irrigation of foley'
 'Yes (for wound dehissance and perineal pain 13 days post-op)'
 'Yes (pre-op, not post op)' 'Yes (for donor site pain)'
 'Yes (for dehiscence)' 'Yes (gross hematuria after 5 mo cysto)'
 'Yes - UC fistula from SPC site' 'Yes (for wound infection)'
 'Yes (right epididymitis)' 'Yes - UTI; thigh abscess' 'yes - retention'
 'Yes (hematuria/retention)' 'Unknown' 'yes (perioperatively)'
 'yes (not tolerating cipro)' 'ER visit before BUR' 'yes (infection)' 0 1]


In [30]:
def clean_er_visits(val):
    if pd.isnull(val):
        return 0
    v = str(val).strip().lower()
    if v.startswith('yes'):
        return 1
    if v.startswith('no'):
        return 0
    if v == "1":
        return 1
    if v == "0":
        return 0
    return 0

In [31]:
df["ER visits"] = df["ER visits"].apply(clean_er_visits).astype(int)

In [32]:
print(df["ER visits"].unique())

[1 0]


In [33]:
df.columns

Index(['Age_calc', 'COPD', 'Diabetes', 'COPD.1', 'Smoker', 'Abx',
       'Transection', 'Failure', 'fu', 'datetofailureorfollowup', 'ER visits',
       'UTI  Post', 'UTI  recurring', 'foley', 'failure date'],
      dtype='object')

In [34]:
num_failures = (df["Failure"] == 1).sum()
print(num_failures)

176


In [35]:
print(df["failure date"].unique())

[nan datetime.datetime(2015, 5, 19, 0, 0)
 datetime.datetime(2004, 11, 16, 0, 0)
 datetime.datetime(2010, 11, 10, 0, 0)
 datetime.datetime(2007, 3, 14, 0, 0) datetime.datetime(2016, 1, 1, 0, 0)
 datetime.datetime(2022, 1, 12, 0, 0)
 datetime.datetime(2020, 11, 16, 0, 0) datetime.datetime(2020, 9, 9, 0, 0)
 'died 7-4-22' datetime.datetime(2021, 4, 16, 0, 0)
 datetime.datetime(2021, 8, 5, 0, 0) datetime.datetime(2022, 4, 6, 0, 0)
 'died 26-3-22' datetime.datetime(2021, 12, 2, 0, 0)
 datetime.datetime(2024, 6, 3, 0, 0) datetime.datetime(2022, 12, 14, 0, 0)
 datetime.datetime(2022, 3, 24, 0, 0) datetime.datetime(2022, 6, 15, 0, 0)
 datetime.datetime(2023, 7, 14, 0, 0) datetime.datetime(2023, 1, 4, 0, 0)
 datetime.datetime(2022, 7, 28, 0, 0) datetime.datetime(2023, 5, 18, 0, 0)
 datetime.datetime(2022, 7, 27, 0, 0) datetime.datetime(2023, 3, 7, 0, 0)
 datetime.datetime(2022, 11, 3, 0, 0)
 datetime.datetime(2022, 11, 30, 0, 0)
 datetime.datetime(2022, 11, 16, 0, 0) datetime.datetime(2023, 1,

In [36]:
#clean failure date
def clean_failure_date(val):
    #check nan vals
    if pd.isnull(val) or val == 0:
        return pd.NaT
    if isinstance(val, (pd.Timestamp, datetime.datetime)):
        return val
    #deal with the ones that say died or sum like that
    if isinstance(val, str):
        if 'died' in val.lower():
            #try and exttract the date
            date_match = re.search(r'\d{1,2}/\d{1,2}/\d{4}', val)
            if date_match:
                date_str = date_match.group(1)
                try:
                    dt = pd.to_datetime(date_str, dayfirst=True, errors='coerce')
                    return dt
                except:
                    return pd.NaT
        else:
                return pd.NaT
    else:
        return pd.to_datetime(val, errors='coerce')
    
    return pd.NaT

In [37]:
df["failure date"] = df["failure date"].apply(clean_failure_date)


In [39]:
df["failure date"].value_counts()

failure date
2015-05-19    1
2023-05-18    1
2023-03-07    1
2022-11-03    1
2022-11-30    1
2022-11-16    1
2023-01-03    1
2023-01-11    1
2023-01-26    1
2023-04-06    1
2024-04-17    1
2023-03-24    1
2023-05-04    1
2023-08-23    1
2023-12-08    1
2023-11-29    1
2023-12-13    1
2022-07-27    1
2022-07-28    1
2004-11-16    1
2023-01-04    1
2010-11-10    1
2007-03-14    1
2016-01-01    1
2022-01-12    1
2020-11-16    1
2020-09-09    1
2021-04-16    1
2021-08-05    1
2022-04-06    1
2021-12-02    1
2024-06-03    1
2022-12-14    1
2022-03-24    1
2022-06-15    1
2023-07-14    1
2023-11-28    1
Name: count, dtype: int64